# 좌표계산을 통한 빈집과 시설의 거리 계산, 그리고 반경 N-km내의 시설의 개수 합산

In [1]:
import pandas as pd
import numpy as np
from functools import reduce
from collections import defaultdict

현재 작업파일

In [2]:
import os
print(os.getcwd())

d:\BAF-25-1-Marketing\전처리_통합


# **가이드** : 새로운 데이터 / 지역에 대해서 볼 경우

1. **새로운 데이터**  
    1-1, 1-2, 2-3, 3-1, 3-3에 새로운 데이터의 경우를 추가하기
      
2. **새로운 지역**  
    1-3, 2-2, 3-2에 새로운 지역을 추가하기

# 1. 사용할 데이터 전처리

## 1) 데이터 불러오기

앞서 전처리한 데이터들을 불러온다

In [ ]:
house=pd.read_csv('결과데이터/빈집데이터_빈집정보알림e_영남호남지역.csv', encoding='utf-8-sig') # 호남, 영남 빈집 데이터


data_conve=pd.read_csv('결과데이터/영남호남_편의점_2024.csv', encoding='cp949') #2.
data_store=pd.read_csv('결과데이터/영남호남_시장정보.csv', encoding='cp949') #3. 
data_hospital=pd.read_csv('결과데이터/영남호남_병원정보.csv', encoding='cp949') #5.
data_school=pd.read_csv('결과데이터/영남호남_초중학교정보.csv', encoding='cp949') #6.
data_lesson=pd.read_csv('결과데이터/영남호남_학원정보_2025.csv', encoding='cp949') #8.
data_police=pd.read_csv('결과데이터/영남호남_경찰서정보_2023.csv', encoding='cp949') #9.
data_bus_station=pd.read_csv("결과데이터/영남호남_버스정류장정보_2024.csv", encoding="cp949") #11.
data_train_station=pd.read_csv('결과데이터/영남호남_기차역정보.csv', encoding='EUC-KR') #14.
data_express_station=pd.read_csv('결과데이터/영남호남_고속버스터미널정보.csv', encoding='EUC-KR') #15.
data_bank=pd.read_csv('결과데이터/영남호남_은행정보.csv', encoding='cp949') #16.
data_rental=pd.read_csv('결과데이터/영남호남_농기계대여소정보.csv', encoding='cp949') #17.
data_farm_market=pd.read_csv('결과데이터/영남호남_농산물직거래판매장정보.csv', encoding='cp949') #18.
data_car_center=pd.read_csv('결과데이터/영남호남_자동차정비업체정보.csv', encoding='cp949') #19.
#5-1. 요양원데이터는 5. 병원 데이터에 병합
data_farmtech_center=pd.read_csv('결과데이터/영남호남_농업기술센터정보.csv', encoding='cp949') #22.
data_town_center=pd.read_csv('결과데이터/영남호남_행정복지센터정보.csv', encoding='cp949') #22.
data_post_office=pd.read_csv('결과데이터/영남호남_우체국정보.csv', encoding='cp949') #23.
data_farm=pd.read_csv('결과데이터/영남호남_농지매물정보.csv', encoding='utf-8-sig') #25.

## 2) "시설이름"열 생성

코드의 획일성을 위해 각 시설의 이름을 "시설이름"열로 통합시킨다.

In [ ]:
data_conve["시설이름"]=data_conve["Conve_Name"] #2.
data_store["시설이름"]=data_store["Store_Name"] #3
data_hospital["시설이름"]=data_hospital["Hospital_Name"] #5
data_school["시설이름"]=data_school["School_Name"] #6
data_lesson["시설이름"]=data_lesson["Lesson_Name"] #8
data_police["시설이름"]=data_police["Office_Name"] #9
data_bus_station["시설이름"]=data_bus_station["Station_Name"] #11
data_train_station["시설이름"]=data_train_station["Name"] #14
data_express_station["시설이름"]=data_express_station["Name"] #15
data_bank["시설이름"]=data_bank["Bank_Name"] #16
data_rental["시설이름"]=data_rental["Office_Name"] #17
data_farm_market["시설이름"]=data_farm_market["Local_Store_Name"] #18
data_car_center["시설이름"]=data_car_center["Center_Name"] #19
#21. 요양원데이터는 5. 병원 데이터에 병합
data_farmtech_center["시설이름"]=data_farmtech_center["Center_Name"] #22
data_town_center["시설이름"]=data_town_center["Center_Name"] #22
data_post_office["시설이름"]=data_post_office["Office_Name"] #23
data_farm["시설이름"]=data_farm["Address"] #25 #농지 이름이 딱히 없으니 상세주소로 대체

## 3) 지역 추출하기

앞서 데이터를 전처리할때 영남, 호남지방만 추출하고 이를 위에 데이터로 불러왔다.  
전북특별자치도는 전부 전라북도로 통일했으므로 이에 주의하자

## 4) 빈집 주소 전처리

중복처리를 하는 이유는, 어차피 각각의 상세한 집주소가 아니라서 주소의 중복이 있기 때문이다.

In [6]:
house["전체주소"]=house["County"].fillna('').str.strip()+" "+house["City"].fillna('').str.strip()+\
                    " "+house["Town"].fillna('').str.strip()+" "+house["Village"].fillna('').str.strip()
house=house[["Latitude","Longitude","전체주소", "County"]].drop_duplicates(subset='전체주소', keep='first').reset_index(drop=True)
house.head(2)

,Latitude,Longitude,전체주소,County
0,35.713900,128.7810,경상북도 청도군 매전면 두곡리,경상북도
1,35.694585,128.8007,경상북도 청도군 매전면 덕산리,경상북도


# 2. 좌표를 이용한 거리 계산

## 1) 하버사인 좌표거리 공식

데이터의 빠른 계산을 위해 벡터로 계산한다

In [7]:
# 벡터화 하버사인 함수
def haversine_vectorized(lat1, lon1, lat2_array, lon2_array):
    R = 6371.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2_array = np.radians(lat2_array)
    lon2_array = np.radians(lon2_array)

    dlat = lat2_array - lat1
    dlon = lon2_array - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2_array) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

거리의 단위는 km이다.

## 2) 빈집과 시설의 거리 계산

In [8]:
def calculate_distance_house(house, data, batch_size=500):
    from collections import defaultdict
    
    # 미리 County별 시설 데이터 캐싱
    county_facility_map = defaultdict(lambda: data)
    for county in ["경상북도", "전라북도", "전라남도", "경상남도"]:
        county_facility_map[county] = data[data['County'] == county]

    house_addrs = house['전체주소'].values
    house_lats = house['Latitude'].values
    house_lons = house['Longitude'].values
    house_counties = house['County'].values
    total = len(house)

    all_batches = []

    for i in range(0, total, batch_size):
        batch_addrs = house_addrs[i:i+batch_size]
        batch_lats = house_lats[i:i+batch_size]
        batch_lons = house_lons[i:i+batch_size]
        batch_counties = house_counties[i:i+batch_size]

        batch_results = []

        for lat1, lon1, addr, county in zip(batch_lats, batch_lons, batch_addrs, batch_counties):
            sub_data = county_facility_map[county]  # 미리 캐싱된 데이터 사용

            sub_data_lats = sub_data['Latitude'].values
            sub_data_lons = sub_data['Longitude'].values
            sub_data_names = sub_data['시설이름'].values

            dists = haversine_vectorized(lat1, lon1, sub_data_lats, sub_data_lons)
            dists = np.round(dists, 2)

            result_row = {'빈집 주소': addr}
            result_row.update(dict(zip(sub_data_names, dists)))  # 딕셔너리 병합

            batch_results.append(result_row)

        all_batches.append(pd.DataFrame(batch_results))

    return pd.concat(all_batches, ignore_index=True)

## 3) 아래 "3.반경 N-km 시설 개수 합산"에 사용할 데이터 생성 및 저장

각각의 데이터에 대해 실행하고, csv파일로 저장하는 것은 데이터의 크기가 크기 때문에 에러가 발생하기 때문이다.

서로 다른 곳에 위치한 시설이지만, 이름이 서로 같아서 추가적으로 계산되는 것이 몇개 있다.  
사소해서 넘어간다 (왜냐하면 다른 '시도'에 있으면 그만큼 거리는 멀어지고, 후에 있을 반경 개수 계산할 때 나가떨어지기 때문이다)

In [ ]:
#2. 편의점 
result_house_conve=calculate_distance_house(house, data_conve)
result_house_conve.to_csv('좌표거리데이터/result_house_conve.csv', index=False, encoding='utf-8-sig')

In [ ]:
#3. 시장
result_house_store=calculate_distance_house(house, data_store)
result_house_store.to_csv('좌표거리데이터/result_house_store.csv', index=False, encoding='utf-8-sig')

In [ ]:
#5. 병원 
result_house_hospital=calculate_distance_house(house, data_hospital)
result_house_hospital.to_csv('좌표거리데이터/result_house_hospital.csv', index=False, encoding='utf-8-sig')

In [ ]:
#6. 초중학교
result_house_school=calculate_distance_house(house, data_school)
result_house_school.to_csv('좌표거리데이터/result_house_school.csv', index=False, encoding='utf-8-sig')

In [ ]:
#8. 학원
result_house_lesson=calculate_distance_house(house, data_lesson)
result_house_lesson.to_csv('좌표거리데이터/result_house_lesson.csv', index=False, encoding='utf-8-sig')

In [ ]:
#9. 경찰서
result_house_police=calculate_distance_house(house, data_police)
result_house_police.to_csv('좌표거리데이터/result_house_police.csv', index=False, encoding='utf-8-sig')

In [ ]:
#11. 버스정류장
result_house_bus_station=calculate_distance_house(house, data_bus_station)
result_house_bus_station.to_csv('좌표거리데이터/result_house_bus_station.csv', index=False, encoding='utf-8-sig')

In [ ]:
#14. 기차역
result_house_train_station=calculate_distance_house(house, data_train_station)
result_house_train_station.to_csv('좌표거리데이터/result_house_train_station.csv', index=False, encoding='utf-8-sig')

In [9]:
#15. 고속버스터미널
result_house_express_station=calculate_distance_house(house, data_express_station)
result_house_express_station.to_csv('좌표거리데이터/result_house_express_station.csv', index=False, encoding='utf-8-sig')

In [ ]:
#16. 은행
result_house_bank=calculate_distance_house(house, data_bank)
result_house_bank.to_csv('좌표거리데이터/result_house_bank.csv', index=False, encoding='utf-8-sig')

In [ ]:
#17. 농기계대여소
result_house_rental=calculate_distance_house(house, data_rental)
result_house_rental.to_csv('좌표거리데이터/result_house_rental.csv', index=False, encoding='utf-8-sig')

In [ ]:
#18. 농산물직거래판매장
result_house_farm_market=calculate_distance_house(house, data_farm_market)
result_house_farm_market.to_csv('좌표거리데이터/result_house_farm_market.csv', index=False, encoding='utf-8-sig')

In [ ]:
#19. 자동차정비업체
result_house_car_center=calculate_distance_house(house, data_car_center)
result_house_car_center.to_csv('좌표거리데이터/result_house_car_center.csv', index=False, encoding='utf-8-sig')

In [ ]:
#22. 농업기술센터
result_house_farmtech_center=calculate_distance_house(house, data_farmtech_center)
result_house_farmtech_center.to_csv('좌표거리데이터/result_house_farmtech_center.csv', index=False, encoding='utf-8-sig')

In [ ]:
#22. 행정복지센터
result_house_town_center=calculate_distance_house(house, data_town_center)
result_house_town_center.to_csv('좌표거리데이터/result_house_town_center.csv', index=False, encoding='utf-8-sig')

In [ ]:
#23. 우체국
result_house_post_office=calculate_distance_house(house, data_post_office)
result_house_post_office.to_csv('좌표거리데이터/result_house_post_office.csv', index=False, encoding='utf-8-sig')

In [ ]:
#25. 농지매물
result_house_farm=calculate_distance_house(house, data_farm)
result_house_farm.to_csv('좌표거리데이터/result_house_farm.csv', index=False, encoding='utf-8-sig')